# NB_hospital_daily_summary

Builds the **gold.hospital_daily_summary** table — one row per hospital per calendar day with test volumes, abnormal/critical rates, and day-over-day trend via LAG. Full rebuild (CREATE OR REPLACE).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresyn.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresyn.gold.hospital_daily_summary AS

WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                      AS summary_date,
    COUNT(*)                                                          AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                     AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                      AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)        AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)        AS critical_test_count
  FROM adb_caresyn.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),

with_rates_and_lag AS (
  SELECT
    da.*,
    ROUND(da.abnormal_test_count * 100.0 / da.total_tests, 2)        AS abnormal_rate_pct,
    ROUND(da.critical_test_count * 100.0 / da.total_tests, 2)        AS critical_rate_pct,
    LAG(da.total_tests)
      OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date)     AS prev_day_total_tests
  FROM daily_agg da
)

SELECT
  wl.hospital_id,
  h.hospital_name,
  wl.summary_date,
  c.day_name,
  c.is_weekend,
  wl.total_tests,
  wl.unique_patients_tested,
  wl.active_doctor_count,
  wl.abnormal_test_count,
  wl.critical_test_count,
  CAST(wl.abnormal_rate_pct   AS DECIMAL(5,2))                       AS abnormal_rate_pct,
  CAST(wl.critical_rate_pct   AS DECIMAL(5,2))                       AS critical_rate_pct,
  wl.prev_day_total_tests,
  CAST(
    CASE
      WHEN wl.prev_day_total_tests IS NULL OR wl.prev_day_total_tests = 0 THEN NULL
      ELSE ROUND((wl.total_tests - wl.prev_day_total_tests) * 100.0 / wl.prev_day_total_tests, 2)
    END
  AS DECIMAL(6,2))                                                   AS test_volume_change_pct

FROM with_rates_and_lag wl

LEFT JOIN adb_caresyn.silver.hospitals h
  ON wl.hospital_id = h.hospital_id

LEFT JOIN adb_caresyn.silver.calendar c
  ON c.date_key = CAST(DATE_FORMAT(wl.summary_date, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresyn.gold.hospital_daily_summary")
record_count = updated_df.count()

dbutils.notebook.exit(str(record_count))